In [1]:
# ===== 1: 설정 =====
import sys
import importlib
from pathlib import Path

PROJECT_DIR = Path.home() / "AI_programming" / "petmint-rag-chatbot"
sys.path.insert(0, str(PROJECT_DIR))

import data.search_scope as search_scope
import data.search_engine as search_engine
import data.rag_context as rag_context
importlib.reload(search_scope)    # 모듈 수정 후 커널 재시작 없이 반영(search_scope가 먼저 — 나머지가 import한다)
importlib.reload(search_engine)
importlib.reload(rag_context)
from data.search_scope import SearchScope

engine = search_engine.SearchEngine()   # Hybrid Search + Reranker + 메타데이터 필터
# client/청크는 engine 것을 재사용(중복 로드 방지). 그래프 키-청크 정합성은 생성 시 assert
builder = rag_context.ContextBuilder(client=engine.client, chunks=engine.chunks)

print("강아지 그래프 노드:", len(builder.graphs["강아지"]))
print("고양이 그래프 노드:", len(builder.graphs["고양이"]))

/home/kopo17/llm_env/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


임베딩 모델 로드 중...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 19089.87it/s]

리랭커 로드 중...


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 5376.73it/s]

검색 엔진 초기화 완료
강아지 그래프 노드: 446
고양이 그래프 노드: 408


In [2]:
# ===== 2: 검색 + 그래프 보강 + 참조 조문/inline 법령 정의 (로직은 data/rag_context.py) =====
# 검색 범위는 SearchScope가 정한다. scope를 안 주면 질문에 적힌 등급·유형으로 의료비 특약만 좁힌다(from_query) —
# 예전처럼 컬렉션 전체에 걸지 않아 보통약관·일반조항 등 등급 없는 청크가 남는다(2026-10-13 수정).
# 고객 가입 정보로 검색하려면 scope=SearchScope.for_policies(species, policies).

def search_with_graph(query: str, species: str, top_k: int = 5, candidate_k: int = 20, scope: SearchScope = None):
    scope = scope or SearchScope.from_query(query, species)
    return rag_context.search_with_graph(engine, builder, query, scope, top_k=top_k, candidate_k=candidate_k)

def build_full_context(query: str, species: str, top_k: int = 3, scope: SearchScope = None) -> list[dict]:
    scope = scope or SearchScope.from_query(query, species)
    return rag_context.build_full_context(engine, builder, query, scope, top_k=top_k)

In [3]:
# ===== 3: 참조 조문 보강 테스트 =====
# 인용 표기는 citation_label 사용("제{article_num}조(...)" 조립은 법령의 "의N"이 빠진다)

test_final = build_full_context("통원 한도가 얼마인가요?", "강아지", top_k=1)
for ref in test_final[0]["reference_contents"]:
    print(f"{ref['citation_label']} — 길이: {len(ref['content']) if ref['content'] else 0}자")


In [4]:
test_vet = build_full_context("동물병원은 어떤 곳을 말하나요?", "강아지", top_k=1)
print("참조:")
for ref in test_vet[0]["reference_contents"]:
    print(f"{ref['citation_label']} — 길이: {len(ref['content']) if ref['content'] else 0}자")

참조:
제1조(보험금의 지급사유) 제2항 — 길이: 718자
수의사법 제2조(정의) — 길이: 146자


In [5]:
test_vet_cat = build_full_context("동물병원은 어떤 곳을 말하나요?", "고양이", top_k=1)
print("참조:")
for ref in test_vet_cat[0]["reference_contents"]:
    print(f"{ref['citation_label']} — 길이: {len(ref['content']) if ref['content'] else 0}자")

참조:
제1조(보험금의 지급사유) 제2항 — 길이: 718자
수의사법 제2조(정의) — 길이: 146자


In [6]:
test_grade = build_full_context("고급형 수술 한도가 얼마인가요?", "강아지", top_k=3)
for r in test_grade:
    print(f"\n[type={r.get('type')}, coverage_type={r.get('coverage_type')}] {r.get('section_name')}")
    print(r["content"][:150])


[type=Ⅰ, coverage_type=고급형] 펫퍼민트 반려견 통원의료비보장 특별약관
회사가 보상하는 비용은 각 항목별 피보험자가 부담한
치료비에서 보험증권에 기재된 자기부담금을 각각 차감한
후, 보험증권에 기재된 보상비율(70%)을 곱한 금액을 아래
에서 정한 금액을 한도로 보상합니다.

      
      항목                    

[type=Ⅱ, coverage_type=고급형] 펫퍼민트 반려견 통원의료비Ⅱ보장 특별약관
회사가 보상하는 비용은 각 항목별 피보험자가 부담한
치료비에서 보험증권에 기재된 자기부담금을 각각 차감한
후, 보험증권에 기재된 보상비율(70%)을 곱한 금액을 아래
에서 정한 금액을 한도로 보상합니다.

      
      항목                    

[type=Ⅲ, coverage_type=고급형] 펫퍼민트 반려견 통원의료비Ⅲ보장 특별약관
 【보험금 지급금액[자기부담금 3만원 예시]】
 ①통원 중 수술을 하지 않은 경우(보상비율 70% 가입,
     MRI,CT 및 내시경처치를 받지 않은 날)
 ·피보험자가 부담한 치료비 33만원
 ·보험금 지급금액
     = [(33만원 - 3만원)×70%, 30만


In [7]:
# 1) 고양이 등급 혼동 테스트
test_cat_grade = build_full_context("고급형 수술 한도가 얼마인가요?", "고양이", top_k=3)
for r in test_cat_grade:
    print(f"\n[type={r.get('type')}, coverage_type={r.get('coverage_type')}] {r.get('section_name')}")
    print(r["content"][:150])

# 2) type 필터 작동 확인
test_type_filter = build_full_context("MRI 검사 비용이 궁금해요(Ⅲ유형)", "강아지", top_k=3)
for r in test_type_filter:
    print(f"[type={r.get('type')}] {r.get('section_name')}")


[type=Ⅰ, coverage_type=고급형] 펫퍼민트 반려묘 통원의료비보장 특별약관
회사가 보상하는 비용은 각 항목별 피보험자가 부담한
치료비에서 보험증권에 기재된 자기부담금을 각각 차감한
후, 보험증권에 기재된 보상비율(70%)을 곱한 금액을 아래
에서 정한 금액을 한도로 보상합니다.

      
      항목                    

[type=Ⅱ, coverage_type=고급형] 펫퍼민트 반려묘 통원의료비Ⅱ보장 특별약관
회사가 보상하는 비용은 각 항목별 피보험자가 부담한
치료비에서 보험증권에 기재된 자기부담금을 각각 차감한
후, 보험증권에 기재된 보상비율(70%)을 곱한 금액을 아래
에서 정한 금액을 한도로 보상합니다.

      
      항목                    

[type=Ⅲ, coverage_type=고급형] 펫퍼민트 반려묘 통원의료비Ⅲ보장 특별약관
 【보험금 지급금액[자기부담금 3만원 예시]】
 ①통원 중 수술을 하지 않은 경우(보상비율 70% 가입,
     MRI,CT 및 내시경처치를 받지 않은 날)
 ·피보험자가 부담한 치료비 33만원
 ·보험금 지급금액
     = [(33만원 - 3만원)×70%, 30만


[type=Ⅲ] 펫퍼민트 반려견 통원의료비Ⅲ보장 특별약관
[type=Ⅲ] 펫퍼민트 반려견 통원의료비Ⅲ보장 특별약관
[type=Ⅲ] 펫퍼민트 반려견 통원의료비Ⅲ보장 특별약관
